# 05 — Pseudo-Labeling (Self-Training)

Iteratively fine-tunes `utils.config.CLASSIFIER_MODEL_NAME` on the labeled
pool, predicts on the unlabeled pool, and absorbs high-confidence
predictions each round. Stops once `target_coverage` (95%) of
the whole pool (seed + unlabeled) has been labeled, the unlabeled pool is
exhausted, or a round absorbs zero new pseudo-labels (model too
underconfident to progress further at this sample size/threshold) —
whichever comes first, up to a `max_iterations` safety cap. Each round
prints total sample size, new high-confidence labels absorbed, and
remaining unlabeled count. Uses a sample size much smaller than the
embedding tasks' `SAMPLE_SIZE` since fine-tuning is far more CPU-expensive
than frozen embedding — this is still the most expensive notebook in the
plan. The full-data final run should be kicked off unattended and budgeted
in hours, not minutes.

**Experiment 1 (2026-08-22):** at the fixed 0.90 confidence threshold and
`CLASSIFIER_SAMPLE_SIZE=150`, this stalled at 0 absorbed pseudo-labels on
the first round (see `docs/semi_supervised_methods.md`) — the DistilBERT
model fine-tuned on only ~152 labeled rows never reaches 90% softmax
confidence on anything, even though its *predictions* are often correct
(82.8% test accuracy). Lowering `confidence_threshold` to 0.60 confirmed
this was the real cause: 4 rounds ran, 72.8% pool coverage, 91.2% label
accuracy, and test accuracy rose to 84.9%.

**Experiment 2:** now testing whether a bigger sample (`PSEUDO_LABEL_SAMPLE_SIZE=400`,
still well short of the historical 500-row timeout) compounds that gain —
scoped to this notebook only, not `utils.config.CLASSIFIER_SAMPLE_SIZE`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_label_quality, evaluate_semisupervised
from utils.modeling import get_predictions, pseudo_label_loop

In [2]:
labeled_df = pd.read_parquet(config.PROCESSED_DIR / "labeled.parquet")
unlabeled_df = pd.read_parquet(config.PROCESSED_DIR / "unlabeled.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

# Scoped to this notebook only (not utils.config.CLASSIFIER_SAMPLE_SIZE,
# which 06_full_supervised_baseline.ipynb also reads) — second experiment,
# now that lowering confidence_threshold alone was confirmed to work.
PSEUDO_LABEL_SAMPLE_SIZE = 400
labeled_sample = stratified_sample(labeled_df, PSEUDO_LABEL_SAMPLE_SIZE, seed=config.SEED)
unlabeled_sample = stratified_sample(unlabeled_df, PSEUDO_LABEL_SAMPLE_SIZE, seed=config.SEED)

overlap = set(unlabeled_sample["text"]) & set(test_clean["text"])
assert len(overlap) == 0, f"{len(overlap)} rows leaked between train pool and test set"
print(f"Labeled sample: {len(labeled_sample)} | Unlabeled sample: {len(unlabeled_sample)} | Test: {len(test_clean)}")
print("No train/test text overlap confirmed.")

Labeled sample: 400 | Unlabeled sample: 400 | Test: 7600
No train/test text overlap confirmed.


In [3]:
final_model, final_tokenizer, current_labeled, history = pseudo_label_loop(
    labeled_sample, unlabeled_sample,
    model_name=config.CLASSIFIER_MODEL_NAME,
    confidence_threshold=0.60, epochs=3,
    target_coverage=0.95, max_iterations=10)

for h in history:
    print(h)

Total sample: 800 | target coverage: 95% | confidence threshold: 0.6


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,0.695495


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,0.679562
100,0.187397


{'iteration': 0, 'total_sample': 800, 'new_labels': 374, 'new_labels_pct_of_total': 0.4675, 'labeled_size': 774, 'coverage': 0.9675, 'unlabeled_size': 26}


In [4]:
pseudo_only = current_labeled.iloc[len(labeled_sample):]
merged = pseudo_only.merge(unlabeled_sample[["text", "true_label"]], on="text", how="left")

label_quality = evaluate_label_quality(
    true_labels=merged["true_label"].to_numpy(),
    pseudo_labels=merged["label"].to_numpy())
print("Pseudo-label quality:", label_quality)

Pseudo-label quality: {'Label Accuracy': 0.9037433155080213, 'Label Macro F1': 0.9063399093124792, 'Coverage': np.float64(1.0)}


In [5]:
test_probs = get_predictions(final_model, final_tokenizer, test_clean["text"].tolist())
test_preds = test_probs.argmax(axis=1)

semisup_results, report, cm = evaluate_semisupervised(
    test_clean["label"].to_numpy(), test_preds, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_pseudo_labeling.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_pseudo_labeling.json", "w") as f:
    json.dump({"test_metrics": semisup_results, "label_quality": label_quality, "history": history},
               f, indent=2)
print("Saved pseudo-labeling results.")

              precision    recall  f1-score   support

       World       0.91      0.88      0.89      1900
      Sports       0.96      0.97      0.96      1900
    Business       0.86      0.81      0.83      1900
    Sci/Tech       0.83      0.90      0.86      1900

    accuracy                           0.89      7600
   macro avg       0.89      0.89      0.89      7600
weighted avg       0.89      0.89      0.89      7600

Saved pseudo-labeling results.
